# 29 · Power Automate flow fundamentals

## Goal

Build two flows — a Dataverse-trigger flow and a button-triggered flow from
the app — and get comfortable with `pac solution unpack`'s flow JSON as
the thing you actually diff, the same way `agents/` YAML is the thing you
diff for the agent.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../apps/renewal-desk-canvas/flows").exists()


## Concept

Flows don't have their own dedicated `pac flow` authoring CLI the way
agents have `pac copilot`. What they have is `pac solution unpack`:
export a solution containing the flow, unpack it, and the cloud flow lands
as a single JSON file (classic workflows unpack separately, under
`workflows/`; cloud flows under a `modernflows/`-shaped folder). That JSON
*is* the flow's definition — triggers, actions, connection references —
and it's what a reviewer actually reads on a PR, the same role
`instructions.md`'s diff plays for the agent.

Two flows, two trigger shapes worth having distinct examples of:
`NotifyOnHighRiskFlag` reacts to a Dataverse row change (mirrors `17`'s
cascading-rule workflow, but as an app-layer flow instead of an
agent-layer workflow — the same "notify + block" cascade, different
runtime); `TriageOnDemand` is button-triggered from the app itself, the
shape `30` extends to call the agent.


## Build


### Build both flows (Power Automate designer), then export and unpack


`NotifyOnHighRiskFlag`: trigger = Dataverse row modified on
`crd_supplierrenewal` where `crd_autorenewalenabled` changes to false;
action = post a Teams message. `TriageOnDemand`: trigger = Power Apps
(manual, called from a button); no actions yet — `30` adds the agent call.


In [ ]:
import subprocess
export = subprocess.run([
    "pac", "solution", "export",
    "--name", "crd-renewal-desk-flows",
    "--path", "../dist/renewal-desk-flows.zip",
    "--managed", "false",
], capture_output=True, text=True)
print(export.returncode)


In [ ]:
from csx.pac import solution_unpack
from pathlib import Path
solution_unpack(Path("../dist/renewal-desk-flows.zip"), Path("../apps/renewal-desk-canvas/flows/_unpacked"))
import shutil
for f in Path("../apps/renewal-desk-canvas/flows/_unpacked").rglob("*.json"):
    if "modernflow" in str(f).lower() or "workflow" in str(f).lower():
        shutil.copy(f, Path("../apps/renewal-desk-canvas/flows") / f.name)
print("flow JSON copied to apps/renewal-desk-canvas/flows/ — this is what you'll diff")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import json
from pathlib import Path
flow_files = list(Path("../apps/renewal-desk-canvas/flows").glob("*.json"))
assert len(flow_files) >= 2, f"expected both flows unpacked, found {len(flow_files)}"
for f in flow_files:
    doc = json.loads(f.read_text())
    print(f.name, "trigger kind:", list(doc.get("properties", {}).get("definition", {}).get("triggers", {}).keys()))


## Cost


In [ ]:
print("Flow authoring/export/unpack doesn't consume Copilot Credits. Flow runs themselves are metered under Power Automate's own consumption model, separate from this repo's credit ledger.")


## Teardown


In [ ]:
print("No teardown — both flows persist; 30 wires TriageOnDemand to call the agent.")
